# NumPy 100 — Problems 41–50

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 41. How to sum a small array faster than np.sum? (★★☆)

In [2]:
import timeit

Z = np.arange(10)
print(np.add.reduce(Z) == np.sum(Z))    # True: same result

t_sum    = timeit.timeit(lambda: np.sum(Z), number=200_000)
t_reduce = timeit.timeit(lambda: np.add.reduce(Z), number=200_000)
print(f"np.sum:        {t_sum:.3f} s")
print(f"np.add.reduce: {t_reduce:.3f} s")   # usually faster for small arrays

True


np.sum:        0.329 s
np.add.reduce: 0.087 s


**Concept:** `np.sum` is a Python wrapper that handles many options before calling `np.add.reduce`. For a *small* array that wrapper overhead dominates, so calling `np.add.reduce(Z)` directly is faster. For large arrays the difference disappears. Exact timings depend on your machine, so measure with `timeit` (or `%timeit` in Jupyter).

**Needed help?** solution  
**Revisit on:** Nov 10

---
### 42. Consider two random arrays A and B, check if they are equal (★★☆)

In [3]:
rng = np.random.default_rng(42)
A = rng.integers(0, 2, 5)
B = rng.integers(0, 2, 5)
print(A, B)

print(np.array_equal(A, B))             # exact: same shape and same values
print(np.allclose(A, B))                # equal within a small tolerance
print(np.array_equal(A, A.copy()))      # True

# the two functions differ in what they accept:
print(np.allclose([1, 1, 1], [1]))      # True  (shapes broadcast)
print(np.array_equal([1, 1, 1], [1]))   # False (shapes must match)
print(np.allclose(0.3, 3 * 0.1))        # True  (tolerance handles float error)

[0 1 1 0 0] [1 0 1 0 0]
False
False
True
True
False
True


**Concept:** `np.array_equal(A, B)` is a strict check: same shape and identical values. `np.allclose(A, B)` allows a small tolerance and lets shapes broadcast, which is what you want for floats. Never use `A == B` as a single answer, because it returns an array of booleans.

**Needed help?** hint  
**Revisit on:** Nov 10

---
### 43. Make an array immutable (read-only) (★★☆)

In [4]:
Z = np.zeros(10)
Z.flags.writeable = False

try:
    Z[0] = 1
except ValueError as e:
    print("ValueError:", e)             # assignment destination is read-only

print(Z.flags.writeable)                # False

ValueError: assignment destination is read-only
False


**Concept:** setting `Z.flags.writeable = False` makes an array read-only. Any attempt to modify it raises a `ValueError`. Use it to protect data from accidental changes. Set it back to `True` to allow edits again (views made from a read-only array are also read-only).

**Needed help?** hint  
**Revisit on:** Nov 10

---
### 44. Consider a random 10x2 matrix representing cartesian coordinates, convert them to polar coordinates (★★☆)

In [5]:
rng = np.random.default_rng(42)
Z = rng.random((10, 2))
X, Y = Z[:, 0], Z[:, 1]

R = np.sqrt(X**2 + Y**2)                # radius
T = np.arctan2(Y, X)                    # angle in radians

print(R)
print(T)
print(np.allclose(R, np.hypot(X, Y)))   # True: np.hypot does the same as sqrt(x²+y²)

[0.88973156 1.10612502 0.98015731 1.0941804  0.46825271 0.99819062
 1.04474828 0.49825053 0.55824451 1.04114037]
[0.51584563 0.68214621 1.47456395 0.80150622 1.29366315 1.19020435
 0.90677675 0.47357778 0.11456822 0.65190816]
True


**Concept:** polar coordinates: radius `r = sqrt(x² + y²)` and angle `theta = arctan2(y, x)`. Use `arctan2(y, x)` rather than `arctan(y / x)`, because it handles all four quadrants and `x = 0` correctly. `Z[:, 0]` and `Z[:, 1]` slice out the columns. `np.hypot(x, y)` is a built-in for the radius.

**Needed help?** hint  
**Revisit on:** Nov 10

---
### 45. Create random vector of size 10 and replace the maximum value by 0 (★★☆)

In [6]:
rng = np.random.default_rng(42)
Z = rng.random(10)
print(Z)

Z[Z.argmax()] = 0
print(Z)

[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735 0.97562235
 0.7611397  0.78606431 0.12811363 0.45038594]
[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735 0.
 0.7611397  0.78606431 0.12811363 0.45038594]


**Concept:** `Z.argmax()` returns the *index* of the maximum, and you assign through that index. If the maximum appears more than once, `argmax` only returns the first one. To replace all of them, use a mask: `Z[Z == Z.max()] = 0`.

**Needed help?** no  
**Revisit on:** not needed

---
### 46. Create a structured array with `x` and `y` coordinates covering the [0,1]x[0,1] area (★★☆)

In [7]:
Z = np.zeros((5, 5), dtype=[("x", float), ("y", float)])
Z["x"], Z["y"] = np.meshgrid(np.linspace(0, 1, 5),
                             np.linspace(0, 1, 5))
print(Z)
print(Z.dtype)
print(Z["x"][0])                        # x values along the first row

[[(0.  , 0.  ) (0.25, 0.  ) (0.5 , 0.  ) (0.75, 0.  ) (1.  , 0.  )]
 [(0.  , 0.25) (0.25, 0.25) (0.5 , 0.25) (0.75, 0.25) (1.  , 0.25)]
 [(0.  , 0.5 ) (0.25, 0.5 ) (0.5 , 0.5 ) (0.75, 0.5 ) (1.  , 0.5 )]
 [(0.  , 0.75) (0.25, 0.75) (0.5 , 0.75) (0.75, 0.75) (1.  , 0.75)]
 [(0.  , 1.  ) (0.25, 1.  ) (0.5 , 1.  ) (0.75, 1.  ) (1.  , 1.  )]]
[('x', '<f8'), ('y', '<f8')]
[0.   0.25 0.5  0.75 1.  ]


**Concept:** a structured array stores named fields (`x`, `y`) in each element. `np.meshgrid(xs, ys)` produces two 2D grids holding every x and y combination, which are assigned to the two fields. The result is a grid of points covering [0, 1] × [0, 1].

**Needed help?** solution  
**Revisit on:** Nov 10

---
### 47. Given two arrays, X and Y, construct the Cauchy matrix C (Cij =1/(xi - yj)) (★★☆)

In [8]:
X = np.arange(8)
Y = X + 0.5

C = 1.0 / np.subtract.outer(X, Y)
print(C.shape)                          # (8, 8)
print(np.isclose(C[2, 3], 1 / (X[2] - Y[3])))   # True: spot-check one entry
print(np.linalg.det(C))

(8, 8)
True
3638.1636371179666


**Concept:** `np.subtract.outer(X, Y)` builds the matrix of all differences `X[i] - Y[j]` in one step (no loops), and `1.0 /` of that is the Cauchy matrix. Every `ufunc` has an `.outer` method. If `X` and `Y` share a value, you would divide by zero and get `inf`.

**Needed help?** solution  
**Revisit on:** Nov 10

---
### 48. Print the minimum and maximum representable values for each numpy scalar type (★★☆)

In [9]:
for dtype in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:8} min={info.min}  max={info.max}")

for dtype in [np.uint8, np.uint16, np.uint32, np.uint64]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:8} min={info.min}  max={info.max}")

for dtype in [np.float32, np.float64]:
    info = np.finfo(dtype)
    print(f"{dtype.__name__:8} min={info.min}  max={info.max}  eps={info.eps}")

int8     min=-128  max=127
int16    min=-32768  max=32767
int32    min=-2147483648  max=2147483647
int64    min=-9223372036854775808  max=9223372036854775807
uint8    min=0  max=255
uint16   min=0  max=65535
uint32   min=0  max=4294967295
uint64   min=0  max=18446744073709551615
float32  min=-3.4028234663852886e+38  max=3.4028234663852886e+38  eps=1.1920928955078125e-07
float64  min=-1.7976931348623157e+308  max=1.7976931348623157e+308  eps=2.220446049250313e-16


**Concept:** `np.iinfo(dtype)` gives the limits of integer types and `np.finfo(dtype)` gives the limits of float types (including `eps`, the smallest gap between 1.0 and the next float). Signed ints range from `-2^(n-1)` to `2^(n-1) - 1`; unsigned ints from 0 to `2^n - 1`. Overflow past these limits wraps around silently, so pick dtypes carefully.

**Needed help?** hint  
**Revisit on:** Nov 10

---
### 49. How to print all the values of an array? (★★☆)

In [10]:
import sys

Z = np.arange(20)

with np.printoptions(threshold=5):              # small threshold: NumPy shortens the output
    print(Z)                                    # [ 0  1  2 ... 17 18 19]

with np.printoptions(threshold=sys.maxsize):    # huge threshold: print everything
    print(Z)

# global version (what the question describes); save and restore the old settings:
old = np.get_printoptions()
np.set_printoptions(threshold=sys.maxsize)
print(np.arange(20))
np.set_printoptions(**old)

[ 0  1  2 ... 17 18 19]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


**Concept:** NumPy abbreviates arrays longer than `threshold` elements (default 1000) with `...`. Set `np.set_printoptions(threshold=sys.maxsize)` to print every value. Prefer the `with np.printoptions(...)` context manager so the change only applies inside the block.

**Needed help?** solution  
**Revisit on:** Nov 10

---
### 50. How to find the closest value (to a given scalar) in a vector? (★★☆)

In [11]:
rng = np.random.default_rng(42)
Z = rng.uniform(0, 1, 10)
z = 0.5

index = np.abs(Z - z).argmin()
closest = Z[index]

print(Z)
print(index, closest)

[0.77395605 0.43887844 0.85859792 0.69736803 0.09417735 0.97562235
 0.7611397  0.78606431 0.12811363 0.45038594]
9 0.45038593789556713


**Concept:** subtract the target and take absolute values to get distances, then `argmin()` gives the index of the smallest distance. `Z[index]` is the closest value. For a multi-dimensional array, use `Z.flat[index]`.

**Needed help?** no  
**Revisit on:** not needed